In [ ]:
import sys, os, pickle
import numpy as np
import h5py
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM
from scipy.stats import linregress
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, './original_shift_code')
import illustris_python as il
import Catalogue
import params_icl as P

## Exploración del catálogo de ensamblaje estelar (Rodriguez-Gomez+2016)

Este notebook explora la estructura real del archivo HDF5 `stellar_assembly.hdf5`.

### Conclusión clave (confirmada)
El catálogo es **AGREGADO por subhalo**, NO por-partícula:
- Un valor por subhalo en cada snapshot
- Acceso: `f['Snapshot_99']['StellarMassInSitu'][sub_idx]`
- **NO existen** campos `ParticleID`, `AssemblyChannel`, `ProgGalaxyMass`, `ProgGalaxyID`

In [ ]:
# ── Estructura del archivo ────────────────────────────────────────────────
with h5py.File(P.SA_FILE, 'r') as f:
    print("Claves top-level:")
    keys = list(f.keys())
    print(f"  {len(keys)} entradas: {keys[:5]} ... {keys[-3:]}")
    
    # Estructura del snapshot 99 (z=0)
    snap99 = f['Snapshot_99']
    print(f"\nCampos en Snapshot_99 ({len(snap99.keys())} campos):")
    for k in snap99.keys():
        arr = snap99[k]
        print(f"  {k:45s} shape={arr.shape}  dtype={arr.dtype}")
    
    # Información del Header si existe
    if 'Header' in f:
        print("\nAtributos del Header:")
        for k, v in f['Header'].attrs.items():
            print(f"  {k}: {v}")

In [ ]:
# ── Unidades y rango de valores ───────────────────────────────────────────
# Los datos de TNG usan 1e10 M_sun/h internamente
# Verificamos comparando con il.groupcat

with h5py.File(P.SA_FILE, 'r') as f:
    s99 = f['Snapshot_99']
    sa_total  = s99['StellarMassTotal'][:]          # unidades internas
    sa_insitu = s99['StellarMassInSitu'][:]
    sa_exsitu = s99['StellarMassExSitu'][:]
    sa_mergers= s99['StellarMassFromCompletedMergers'][:]
    sa_ongoing= s99['StellarMassFromOngoingMergers'][:]
    sa_maj_c  = s99['StellarMassFromCompletedMergersMajor'][:]
    sa_maj_ong= s99['StellarMassFromOngoingMergersMajor'][:]
    sa_min_c  = s99['StellarMassFromCompletedMergersMajorMinor'][:] - sa_maj_c
    sa_flyby  = s99['StellarMassFromFlybys'][:]

print(f"N_subhalos en snap 99: {len(sa_total):,}")
print(f"\nRango de StellarMassTotal (unidades internas):")
mask_pos = sa_total > 0
print(f"  min = {sa_total[mask_pos].min():.4e}")
print(f"  max = {sa_total[mask_pos].max():.4e}")
print(f"  N subhalos con M_total > 0: {mask_pos.sum():,}")

# Verificar si las unidades son 1e10 M_sun/h (comparar con groupcat)
Header = il.groupcat.loadHeader(P.basePath, P.SNAP)
print(f"\nTNG h = {P.h} → factor UM = {P.UM:.4e} M_sun por unidad interna")
print(f"Max M_total SA = {sa_total.max()*P.UM:.3e} M_sun")
print(f"(Esperado para BCG de cúmulo masivo: ~1e12–1e13 M_sun)")

In [ ]:
# ── Verificación contra il.groupcat ──────────────────────────────────────
# Comparamos StellarMassTotal del SA con SubhaloMassType[:,4] del groupcat
# para confirmar las unidades y el índice correcto

subhalos_mass = il.groupcat.loadSubhalos(P.basePath, P.SNAP, fields=['SubhaloMassType'])
sub_mstar = subhalos_mass['SubhaloMassType'][:, 4]  # masa estelar TNG interna

# Para los primeros 10 subhalos con masa > 0
ok = (sub_mstar > 0) & (sa_total > 0)
idx_test = np.where(ok)[0][:10]

print("Comparación SubhaloMassType[:,4] vs SA StellarMassTotal (unidades internas):")
print(f"{'idx':>8}  {'GroupCat':>14}  {'SA_Total':>14}  {'Ratio':>8}")
for i in idx_test:
    ratio = sa_total[i] / sub_mstar[i] if sub_mstar[i] > 0 else np.nan
    print(f"{i:8d}  {sub_mstar[i]:14.4e}  {sa_total[i]:14.4e}  {ratio:8.4f}")

print("\nSi Ratio ≈ 1.0 → SA indexado por sub_idx, mismas unidades que groupcat")

In [ ]:
# ── Fracciones de componentes: resumen estadístico ────────────────────────
m_sa = sa_total * P.UM  # M_sun

# Solo subhalos con masa estelar significativa
mask_sig = m_sa > 1e8
f_insitu_all  = np.where(mask_sig, sa_insitu  / sa_total, np.nan)
f_exsitu_all  = np.where(mask_sig, sa_exsitu  / sa_total, np.nan)
f_mergers_all = np.where(mask_sig, sa_mergers / sa_total, np.nan)
f_ongoing_all = np.where(mask_sig, sa_ongoing / sa_total, np.nan)
f_maj_c_all   = np.where(mask_sig, sa_maj_c   / sa_total, np.nan)
f_flyby_all   = np.where(mask_sig, sa_flyby   / sa_total, np.nan)

print(f"Subhalos con M_* > 1e8 M_sun: {mask_sig.sum():,}")
print()
print(f"{'Campo':<35}  {'Mediana':>8}  {'σ':>8}  {'Min':>8}  {'Max':>8}")
print('-'*70)
for label, arr in [
    ('In situ',                  f_insitu_all),
    ('Ex situ (total)',          f_exsitu_all),
    ('Mergers completados',      f_mergers_all),
    ('  Mayor (>1:4)',           f_maj_c_all),
    ('  Menor (1:4–1:10)',       np.where(mask_sig, sa_min_c/sa_total, np.nan)),
    ('Ongoing / stripped',       f_ongoing_all),
    ('Flybys',                   f_flyby_all),
]:
    med = np.nanmedian(arr)
    std = np.nanstd(arr)
    mn  = np.nanmin(arr)
    mx  = np.nanmax(arr)
    print(f"{label:<35}  {med:8.4f}  {std:8.4f}  {mn:8.4f}  {mx:8.4f}")

print(f"\nSuma mediana (insitu+mergers+ongoing+flyby): "
      f"{np.nanmedian(f_insitu_all)+np.nanmedian(f_mergers_all)+np.nanmedian(f_ongoing_all)+np.nanmedian(f_flyby_all):.4f}")

In [ ]:
# ── Test con los BCGs del catálogo ICL ───────────────────────────────────
try:
    with h5py.File(P.CATALOG_OUT, 'r') as f:
        bcg_sub_idx = f['bcg_sub_idx'][:]
        M200c       = f['M200c_Msun'][:]
    n_cl = len(bcg_sub_idx)
    
    # Fracciones SA para los BCGs
    sa_tot_bcg = sa_total[bcg_sub_idx]
    ok_m = sa_tot_bcg > 0
    f_in_bcg  = np.where(ok_m, sa_insitu[bcg_sub_idx]  / sa_tot_bcg, np.nan)
    f_ex_bcg  = np.where(ok_m, sa_exsitu[bcg_sub_idx]  / sa_tot_bcg, np.nan)
    f_mg_bcg  = np.where(ok_m, sa_mergers[bcg_sub_idx] / sa_tot_bcg, np.nan)
    f_og_bcg  = np.where(ok_m, sa_ongoing[bcg_sub_idx] / sa_tot_bcg, np.nan)
    
    print(f"BCGs analizados: {n_cl}")
    print(f"\nFracciones SA para los {n_cl} BCGs:")
    print(f"  f_insitu  mediana={np.nanmedian(f_in_bcg):.3f}  σ={np.nanstd(f_in_bcg):.3f}")
    print(f"  f_exsitu  mediana={np.nanmedian(f_ex_bcg):.3f}  σ={np.nanstd(f_ex_bcg):.3f}")
    print(f"  f_mergers mediana={np.nanmedian(f_mg_bcg):.3f}  σ={np.nanstd(f_mg_bcg):.3f}")
    print(f"  f_ongoing mediana={np.nanmedian(f_og_bcg):.3f}  σ={np.nanstd(f_og_bcg):.3f}")
    
    # Plot rápido
    lM = np.log10(M200c)
    fig, ax = plt.subplots(figsize=(8,5))
    for f_arr, col, lbl in [(f_in_bcg,'#E91E63','In situ'),
                             (f_mg_bcg,'#9C27B0','Mergers'),
                             (f_og_bcg,'#00BCD4','Stripped')]:
        ok = np.isfinite(f_arr)
        ax.scatter(lM[ok], f_arr[ok], color=col, label=lbl, s=20, alpha=0.8)
    ax.set_xlabel(r'$\log M_{200c}$')
    ax.set_ylabel('Fracción de masa estelar')
    ax.set_title('Fracciones SA para los BCGs (BCG+ICL total)')
    ax.legend(); plt.tight_layout(); plt.show()
except FileNotFoundError:
    print(f"Catálogo ICL no encontrado en {P.CATALOG_OUT}.")
    print("Ejecuta primero 00_catalogo.ipynb")

## Resumen de la estructura del catálogo SA

### Lo que SÍ tenemos
```python
# Acceso correcto (por índice de subhalo)
with h5py.File(SA_FILE) as f:
    M_insitu  = f['Snapshot_99']['StellarMassInSitu'][bcg_sub_idx[i]]  * UM  # M_sun
    M_exsitu  = f['Snapshot_99']['StellarMassExSitu'][bcg_sub_idx[i]]  * UM
    M_mergers = f['Snapshot_99']['StellarMassFromCompletedMergers'][bcg_sub_idx[i]] * UM
    M_ongoing = f['Snapshot_99']['StellarMassFromOngoingMergers'][bcg_sub_idx[i]]   * UM
    # → fracciones para el subhalo central COMPLETO (= BCG+ICL en SUBFIND)
```

### Lo que NO tenemos (requeriría SA por-partícula)
- Por qué partícula es in situ vs ex situ → separación espacial BCG vs ICL por componente
- ID del subhalo progenitor de cada partícula (`ProgGalaxyID`)
- Masa del progenitor por partícula (`ProgGalaxyMass`)

### Equivalencias con Mayes+2026
| Campo SA | Componente Mayes+2026 |
|----------|----------------------|
| `StellarMassInSitu` | In situ |
| `StellarMassFromCompletedMergers` | Mergers completados |
| `StellarMassFromCompletedMergersMajor` | Mergers mayores completados |
| `StellarMassFromOngoingMergers` | Stripped / sobrevivientes |
| `StellarMassExSitu` | Total ex situ (= mergers + ongoing + flybys) |
